# Context

In this notebook we will 

# Load packages

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# from scipy.stats import chi2_contingency
import warnings

warnings.filterwarnings("ignore")
pd.options.display.max_columns = 100
pd.options.display.max_rows = 100

# Load data

using relative paths

In [ ]:
filename = "EDA_regression.ipynb"  # Current file name
print(f"Current file name: {filename}\n")
print(f"Current absolute path: {os.getcwd()}\n")

# Specify the paths, relative to the current file
ACTUAL_DIR = os.path.dirname(os.path.abspath(filename))
BASE_DIR = os.path.dirname(ACTUAL_DIR)
DATA_DIR = os.path.join(BASE_DIR, "data")
OUTPUT_DIR = os.path.join(DATA_DIR, "output_data")

print(f"BASE_DIR: {BASE_DIR}")
print(f"DATA_DIR: {DATA_DIR}")
print(f"OUTPUT_DIR: {OUTPUT_DIR}")

In [ ]:
df = pd.read_csv(os.path.join(OUTPUT_DIR, "suertes_clean.csv")).iloc[:, 1:]

df.head(5)

In [ ]:
df.info()

In [ ]:
# Estilos
sns.set(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

In [ ]:
# Análisis exploratorio de datos

# 1. Resumen estadístico de variables numéricas
print("Resumen estadístico:")
display(df.describe())

df.info()

In [ ]:
# 2. Variables categóricas: conteo de frecuencias
cat_cols = df.select_dtypes(include='object').columns
for col in cat_cols:
    print(f"\nDistribución de: {col}")
    display(df[col].value_counts())

In [ ]:
# 3. Histogramas para variables numéricas
num_cols = df.select_dtypes(include=['float64', 'int64']).columns

# Graficar de a 10 variables por bloque
for i in range(0, len(num_cols), 10):
    subset = num_cols[i:i+10]
    df[subset].hist(bins=30, figsize=(20, 10))
    plt.suptitle(f"Distribución de variables numéricas ({i+1} a {i+len(subset)})", fontsize=16)
    plt.tight_layout(rect=[0, 0, 1, 0.96])  # para que no se sobreponga el título
    plt.show()


In [ ]:
# ANÁLISIS BIVARIADO

# 4. Correlación entre variables numéricas
cor_matrix = df[num_cols].corr()

plt.figure(figsize=(16, 12))
sns.heatmap(cor_matrix, cmap='coolwarm', annot=False)
plt.title("Mapa de calor de correlaciones")
plt.show()

In [ ]:
# Gráfico de barras para el análisis de correlación con la variable objetivo

corr_target = cor_matrix['tch'].drop('tch') 

plt.figure(figsize=(10, 8))
corr_target.sort_values().plot(kind='barh', color='purple')
plt.title('Correlación de características con la variable objetivo (tch)')
plt.xlabel('Correlación')
plt.ylabel('Características')
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# 5. Boxplots de variables categóricas vs. TCH (rendimiento)
target = 'tch'
cat_to_plot = ['zona', 'variedad', 'producto', 'tipo_quema', 't_corte']

for col in cat_to_plot:
    if col in df.columns:
        plt.figure(figsize=(30, 5))
        sns.boxplot(data=df, x=col, y=target)
        plt.title(f"{target} vs {col}")
        plt.xticks(rotation=45)
        plt.show()

In [ ]:
# 6. Scatterplots de algunas variables numéricas con TCH
import math

target = 'tch'
num_cols = df.select_dtypes(include=['float64', 'int64']).columns.drop(['tch'])

# Bloques de 6 gráficos
cols_per_fig = 2
rows_per_fig = 3
plots_per_fig = cols_per_fig * rows_per_fig

for i in range(0, len(num_cols), plots_per_fig):
    subset = num_cols[i:i + plots_per_fig]
    fig, axes = plt.subplots(rows_per_fig, cols_per_fig, figsize=(15, 12))
    axes = axes.flatten()
    
    for j, col in enumerate(subset):
        sns.scatterplot(data=df, x=col, y=target, alpha=0.4, ax=axes[j])
        axes[j].set_title(f"{target} vs {col}", fontsize=10)
    
    # Ocultar ejes vacíos (si hay)
    for k in range(j + 1, len(axes)):
        axes[k].axis("off")

    plt.tight_layout()
    plt.suptitle("Scatterplots: Variables numéricas vs. TCH", fontsize=16, y=1.02)
    plt.show()


### **IMPUTACIONES**


**GRUPO 1: Fertilizantes / productos aplicados (nulos > 80%)**
- urea_46pct_, mez, microzinc, boro_granul_, nito_xtend, sul_amonio, nitrax_s, vinaza

Estrategia aplicada:
- Si al menos un fertilizante fue aplicado (aplico_alguno = 1).
- Si otro fertilizante estaba nulo, se imputó con la media del grupo zona + producto + periodo.
- Si ni siquiera eso fue posible, se imputó con 0 (realmente no se aplicó).

Se crearon columnas binarias como mez_aplicado, vinaza_aplicado, etc., para indicar si cada fertilizante fue aplicado o no.

In [ ]:
# Crear copia del dataframe para trabajar
df_fe = df.copy()

In [ ]:
# Lista de columnas de fertilizantes/productos
fertilizer_cols = [
    'urea_46pct_', 'mez', 'microzinc', 'boro_granul_',
    'nito_xtend', 'sul_amonio', 'nitrax_s', 'vinaza'
]

In [ ]:
# Crear columna auxiliar que indique si al menos uno fue aplicado
df_fe['aplico_alguno'] = df_fe[fertilizer_cols].notnull().any(axis=1).astype(int)

In [ ]:
# Paso 2: Imputación condicional
for col in fertilizer_cols:
    # Crear columna de aplicación específica
    df_fe[f"{col}_aplicado"] = 0  # valor por defecto (no aplicado)

    # Marcar como aplicado si tiene valor registrado
    df_fe.loc[df_fe[col].notnull(), f"{col}_aplicado"] = 1

    # Imputar si está nulo pero se aplicó alguno en la fila
    mask = df_fe[col].isnull() & (df_fe['aplico_alguno'] == 1)

    df_fe.loc[mask, col] = (
        df_fe.groupby(['zona', 'producto', 'periodo'])[col]
        .transform(lambda x: x.fillna(x.mean()))
    )[mask]

    # Si sigue siendo NaN (nadie aplicó nada), entonces es cero
    df_fe[col] = df_fe[col].fillna(0)

    # Actualizar columna aplicada
    df_fe[f"{col}_aplicado"] = (df_fe[col] > 0).astype(int)

# Eliminar columna auxiliar
df_fe.drop(columns='aplico_alguno', inplace=True)

In [ ]:
df_fe.head(5)

**GRUPO 2: Variables climáticas (~78% nulos) / temp__media_0_3, precipitacion_0_3, evaporacion_ciclo, etc.**


También codigo_estacion, que es clave para imputarlas.

Estrategia a usar:
- Construir una nueva variable categorica teniendo en cuenta la edad de los cultivos (rango de edad del cultivo)
- Agrupar por periodo, zona, cerca_de y rango_edad_ult_cos
- Imputar usando el promedio dentro de esa agrupación.


In [ ]:
# 1. Calcular el mínimo redondeado hacia abajo
min_edad = np.floor(df_fe['edad_ult_cos'].min())

# 2. Calcular el percentil 99 y redondear hacia arriba
p99 = np.percentile(df_fe['edad_ult_cos'].dropna(), 99)
max_edad = np.ceil(p99)

# 3. Crear bins de 2 en 2 desde el mínimo hasta el percentil 99
bins = list(np.arange(min_edad, max_edad, 2))
bins.append(np.inf)  # Para capturar valores mayores al percentil 99

# 4. Crear etiquetas para los rangos
labels = [f"{int(bins[i])}-{int(bins[i+1]-1)} meses" for i in range(len(bins)-2)]
labels.append(f">= {int(max_edad)} meses")  # Última categoría para outliers

# 5. Categorizar la columna
df_fe['rango_edad_ult_cos'] = pd.cut(df_fe['edad_ult_cos'], bins=bins, labels=labels, right=False)

In [ ]:
climate_cols = [
    'oscilacion_temp_ciclo', 'temp__media_0_3', 'temp__media_ciclo', 'temp_max_ciclo',
    'temp_min_ciclo', 'humedad_rel_media_0_3', 'humedad_rel_media_ciclo',
    'oscilacion_temp_med_0_3', 'radicion_solar_0_3', 'radiacion_solar_ciclo',
    'precipitacion_0_3', 'precipitacion_ciclo', 'evaporacion_0_3', 'evaporacion_ciclo'
]

for col in climate_cols:
    df_fe[col] = df_fe.groupby(['periodo','zona','cerca_de','rango_edad_ult_cos'])[col].transform(lambda x: x.fillna(x.mean()))

**GRUPO 3: Manejo del cultivo (num_riegos, m3_riego, ult_riego, ddult_riego) y fechas (dosis_madurante, semanas_mad_, fec_madur_)**

- Si num_riegos es nulo pero hubo lluvias altas, quizá no se necesitó riego. Por ende, se rellena con 0. 

Para el madurante se tiene que:

Verificar si hay evidencia de que sí se aplicó algo, con base en:

- Producto contenga la palabra "madurante" o "regulador" (o una lista de nombres específicos).
- Si fec_madur_ y semanas_mad_ tienen datos (entonces, hubo aplicación).

Si dosis_madurante está nulo pero hay señales de que se aplicó:
- Imputar con media del grupo (zona, producto, periodo).
- O con un valor típico.  

In [ ]:
# Imputación de riegos

riego_cols = ['num_riegos', 'm3_riego', 'ddult_riego']
for col in riego_cols:
    df_fe[f"{col}_registrado"] = df_fe[col].notnull().astype(int)
    df_fe[col] = df_fe[col].fillna(0)

df_fe['ult_riego'] = df_fe['ult_riego'].fillna('Sin riego')



# Imputación de dosis_madurante
# Paso 1: Identificar si hay evidencia de que se aplicó un madurante
df_fe['madurante_probable'] = (
    df_fe['producto'].str.contains('madurante|regulador', case=False, na=False) |
    df_fe['semanas_mad_'].notnull() |
    df_fe['fec_madur_'].notnull()
).astype(int)

# Paso 2: Imputar variables relacionadas con madurantes si hay evidencia de aplicación
mad_cols = ['dosis_madurante', 'semanas_mad_']

for col in mad_cols:
    # Crear columna binaria de aplicación
    df_fe[f"{col}_aplicado"] = 0

    # Marcar como aplicado si hay valor
    df_fe.loc[df_fe[col].notnull(), f"{col}_aplicado"] = 1

    # Si hay evidencia de aplicación pero el valor está nulo, imputar con la media del grupo
    mask = df_fe[col].isnull() & (df_fe['madurante_probable'] == 1)

    df_fe.loc[mask, col] = (
        df_fe.groupby(['zona', 'producto', 'periodo'])[col]
        .transform(lambda x: x.fillna(x.mean()))
    )[mask]

    # Si sigue siendo nulo (sin evidencia de aplicación), imputar con 0
    df_fe[col] = df_fe[col].fillna(0)

    # Actualizar columna binaria de aplicación
    df_fe[f"{col}_aplicado"] = (df_fe[col] > 0).astype(int)

# Eliminar la columna auxiliar
df_fe.drop(columns='madurante_probable', inplace=True)


**Grupo 4: Lluvias (nulos ~37%) / Lluvias por tramos: lluvias_0__3, lluvias_ciclo, etc**

Estrategia:
- Construir una nueva variable categorica teniendo en cuenta la edad de los cultivos (rango de edad del cultivo)
- Imputar dentro de periodo, zona, cerca_de, rango_edad_cult_cos

In [ ]:
rain_cols = [
    'lluvias_2_meses_ant_', 'lluvias_0__3', 'lluvias_tres_a_seis',
    'lluvias_seis_a_nueve', 'lluvias_9_a_cos', 'lluvias_ciclo'
]

for col in rain_cols:
    df_fe[col] = df_fe.groupby(['periodo','zona','cerca_de','rango_edad_ult_cos'])[col].transform(lambda x: x.fillna(x.mean()))

In [ ]:
df_fe['lluvias_ciclo'].isnull().sum()

**Grupo 5: Cultivo Orgánico o no y Suelo**

Estrategia:

- El tipo de cultivo (orgánico o no) no depende tanto del lote o zona, sino más de la decisión de manejo de la hacienda. Así que tiene mucho más sentido imputar cult_organico por la combinación de zona-hacienda.
- Suelo (categórica): imputar por suerte o hacienda.
- Imputación de 'dist_km' por hacienda: Las suertes de una misma hacienda suelen estar en una misma zona geográfica, por ende sus distancias al punto de entrega son similares.

In [ ]:
# Imputación de 'cult_organico' por zona y hacienda
df_fe['cult_organico'] = df_fe.groupby(['zona', 'hacienda'])['cult_organico'].transform(
    lambda x: x.fillna(x.mode().iloc[0]) if not x.mode().empty else x
)

In [ ]:
# Imputación de 'suelo' por suerte (relación directa con la ubicación del lote)
df_fe['suelo'] = df_fe.groupby('suerte')['suelo'].transform(
    lambda x: x.fillna(x.mode().iloc[0]) if not x.mode().empty else x
)

In [ ]:
# Imputación de 'dist_km' por hacienda
df_fe['dist_km'] = df_fe.groupby('hacienda')['dist_km'].transform(lambda x: x.fillna(x.mode()))

In [ ]:
df_fe.head(5)
df_fe.info()
df_fe.describe()
df_fe.shape